In [ ]:
# python /code/Mob/A_0_Shizhen_process/total_pipeline.py
import pandas as pd
from glob import glob
import os
import multiprocessing
from functools import partial
from tqdm import tqdm
import gc
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# --- Configuration & Constants ---
# 路径配置
BASE_DIR = "/nas/houce/UK_mobility"
INPUT_DATA_PATH_TEMPLATE = os.path.join(BASE_DIR, "processed_data", "*_{year}_*_processed.csv")
OUTPUT_DATA_PATH = os.path.join(BASE_DIR, "data_20251226_new") # 统一输出路径
HOME_WORK_LOC_DIR = os.path.join(OUTPUT_DATA_PATH, "home_workplace_location")

# 列名常量
DEVICE_ID_COL = 'device_aid'
DURATION_COL = 'duration_location_hour'
LAT_COL = 'latitude'
LON_COL = 'longitude'
LOCATION_COLS = [DEVICE_ID_COL, LAT_COL, LON_COL]

# 数据类型优化字典
DTYPE_DICT = {
    DEVICE_ID_COL: 'str',
    LAT_COL: 'float32',
    LON_COL: 'float32',
    DURATION_COL: 'float32',
    'showed_hours_today': 'float32',
    'home': 'int8',
    'workplace': 'int8'
}

# --- Part 1: Daily Home/Workplace Identification ---
# (保持原有的高效逻辑不变)
def identify_locations(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        df['home'] = 0; df['workplace'] = 0
        return df
    
    max_durations = df.groupby(DEVICE_ID_COL)[DURATION_COL].transform('max')
    df['home'] = (df[DURATION_COL] == max_durations).astype('int8')

    is_not_home = df['home'] == 0
    if is_not_home.any():
        max_work_map = df[is_not_home].groupby(DEVICE_ID_COL)[DURATION_COL].max()
        df['max_work_dur'] = df[DEVICE_ID_COL].map(max_work_map)
        df['workplace'] = (is_not_home & (df[DURATION_COL] == df['max_work_dur'])).astype('int8')
        df.drop(columns=['max_work_dur'], inplace=True)
    else:
        df['workplace'] = 0
        
    df['workplace'] = df['workplace'].fillna(0).astype('int8')
    return df

def process_file(file_path: str, save_path: str, showed_hours_threshold: int = None):
    try:
        # 尝试使用 pyarrow 引擎加速，失败则回退
        engine = 'pyarrow' if pd.__version__ >= '1.4.0' else 'c'
        try:
            df = pd.read_csv(file_path, dtype=DTYPE_DICT, engine=engine)
        except:
            df = pd.read_csv(file_path, dtype=DTYPE_DICT)

        if 'showed_hours_today' in df.columns:
            if showed_hours_threshold is not None:
                df = df[df['showed_hours_today'] >= showed_hours_threshold]
        if 'hours' in df.columns:
            df.drop(columns=['hours'], inplace=True)
        df.drop_duplicates(inplace=True)

        if df.empty: return

        df_final = identify_locations(df)

        file_name_base = os.path.basename(file_path).replace('_processed.csv', '')
        combined_output_path = os.path.join(save_path, 'home_workplace', f'{file_name_base}_home_workplace.csv')
        os.makedirs(os.path.dirname(combined_output_path), exist_ok=True)
        df_final.to_csv(combined_output_path, index=False)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

def run_parallel_processing(files_to_process: list, save_path: str):
    if not files_to_process: return
    os.makedirs(save_path, exist_ok=True)
    worker_func = partial(process_file, save_path=save_path)
    num_processes = max(1, min(multiprocessing.cpu_count() - 2, len(files_to_process)))
    print(f"Starting parallel processing of {len(files_to_process)} files...")
    with multiprocessing.Pool(processes=num_processes) as pool:
        list(tqdm(pool.imap_unordered(worker_func, files_to_process), total=len(files_to_process)))

# --- Part 2: Yearly Aggregation ---
# (保持原逻辑不变)
def process_yearly_data_optimized(year: int, base_path: str, home_only: bool, showed_hours_threshold: int = None):
    print(f"\n--- Starting yearly processing for {year} ---")
    file_pattern = os.path.join(base_path, 'home_workplace', f'*_{year}*_home_workplace.csv')
    file_list = sorted(glob(file_pattern))
    if not file_list: return None

    CHUNK_SIZE = 50
    aggregated_chunks = []
    
    for i in tqdm(range(0, len(file_list), CHUNK_SIZE)):
        chunk_files = file_list[i : i + CHUNK_SIZE]
        chunk_dfs = []
        for f in chunk_files:
            try:
                temp_df = pd.read_csv(f, usecols=LOCATION_COLS + [DURATION_COL, 'showed_hours_today'], dtype=DTYPE_DICT)
                if showed_hours_threshold is not None:
                    temp_df = temp_df[temp_df['showed_hours_today'] >= showed_hours_threshold]
                chunk_dfs.append(temp_df)
            except: continue
        
        if chunk_dfs:
            chunk_concat = pd.concat(chunk_dfs, ignore_index=True)
            chunk_agg = chunk_concat.groupby(LOCATION_COLS, as_index=False)[DURATION_COL].sum()
            aggregated_chunks.append(chunk_agg)
            del chunk_concat, chunk_dfs, chunk_agg
            gc.collect()

    if not aggregated_chunks: return None

    print("Finalizing aggregation...")
    total_hours_df = pd.concat(aggregated_chunks, ignore_index=True)
    total_hours_df = total_hours_df.groupby(LOCATION_COLS, as_index=False)[DURATION_COL].sum()
    
    home_output_path = os.path.join(base_path, 'home_workplace_location', f'home_location_{year}.csv')
    idx = total_hours_df.groupby(DEVICE_ID_COL)[DURATION_COL].idxmax()
    home_locations = total_hours_df.loc[idx].reset_index(drop=True)
    os.makedirs(os.path.dirname(home_output_path), exist_ok=True)
    home_locations.to_csv(home_output_path, index=False)
    
    home_locations['home_flag'] = 1
    total_hours_df = total_hours_df.merge(home_locations[[DEVICE_ID_COL, LAT_COL, LON_COL, 'home_flag']], on=LOCATION_COLS, how='left')
    total_hours_df['home_flag'].fillna(0, inplace=True)

    if not home_only:
        workplace_output_path = os.path.join(base_path, 'home_workplace_location', f'workplace_location_{year}.csv')
        non_home_df = total_hours_df[total_hours_df['home_flag'] == 0]
        if not non_home_df.empty:
            idx_work = non_home_df.groupby(DEVICE_ID_COL)[DURATION_COL].idxmax()
            work_locations = non_home_df.loc[idx_work].reset_index(drop=True)
            work_locations.to_csv(workplace_output_path, index=False)

    print(f"--- Yearly processing for {year} finished ---")
    return total_hours_df 

# --- Part 3: Weekday/Weekend Analysis ---
# (保持原逻辑不变)
def analyze_and_save_day_counts_auto(year: int, base_path: str):
    print(f"Analyzing day counts for {year}...")
    file_pattern = os.path.join(base_path, 'home_workplace', f'*_{year}*_home_workplace.csv')
    file_list = glob(file_pattern)
    if not file_list: return None

    chunk_counts = []
    CHUNK_SIZE = 100
    for i in tqdm(range(0, len(file_list), CHUNK_SIZE)):
        chunk_files = file_list[i : i + CHUNK_SIZE]
        dfs = []
        for f in chunk_files:
            try:
                tmp = pd.read_csv(f, usecols=[DEVICE_ID_COL, 'day'], dtype={DEVICE_ID_COL: 'str', 'day': 'str'})
                dfs.append(tmp)
            except: continue
        if dfs:
            df_chunk = pd.concat(dfs, ignore_index=True).drop_duplicates()
            chunk_counts.append(df_chunk)
    
    if not chunk_counts: return None

    full_days_df = pd.concat(chunk_counts, ignore_index=True).drop_duplicates()
    full_days_df['dt'] = pd.to_datetime(full_days_df['day'], format='%Y%m%d', errors='coerce')
    full_days_df['is_weekend'] = (full_days_df['dt'].dt.dayofweek >= 5).astype(int)
    
    day_counts = full_days_df.groupby([DEVICE_ID_COL, 'is_weekend']).size().unstack(fill_value=0).reset_index()
    day_counts.rename(columns={0: 'day_count_weekday', 1: 'day_count_weekend'}, inplace=True)
    
    for col in ['day_count_weekday', 'day_count_weekend']:
        if col not in day_counts.columns: day_counts[col] = 0
            
    day_counts['day_count_all'] = day_counts['day_count_weekday'] + day_counts['day_count_weekend']
    
    output_file = os.path.join(base_path, 'home_workplace_location', f'weekday_weekend_count_{year}.csv')
    day_counts.to_csv(output_file, index=False)
    print(f"Saved automated day counts to {output_file}")
    return day_counts

# --- Part 4: Final Classification & Analysis (New & Optimized) ---

def get_valid_devices_cross_year(day_counts_2020: pd.DataFrame, day_counts_2021: pd.DataFrame, min_days: int = 4):
    """
    Identifies devices present in both years with sufficient data.
    """
    print("Identifying valid devices across 2020 and 2021...")
    merged = day_counts_2020.merge(day_counts_2021, on=DEVICE_ID_COL, how='outer', suffixes=('_2020', '_2021'))
    
    # Fill NA with 0 for calculation
    merged[['day_count_all_2020', 'day_count_all_2021']] = merged[['day_count_all_2020', 'day_count_all_2021']].fillna(0)
    
    # Filter logic
    valid_mask = (merged['day_count_all_2020'] >= min_days) & (merged['day_count_all_2021'] >= min_days)
    valid_devices = set(merged.loc[valid_mask, DEVICE_ID_COL].unique())
    
    output_path = os.path.join(HOME_WORK_LOC_DIR, "weekday_weekend_count_2020_2021.csv")
    merged['device_aid_keep'] = valid_mask.astype(int)
    merged.to_csv(output_path, index=False)
    
    print(f"Found {len(valid_devices)} valid devices. Saved summary to {output_path}")
    return valid_devices

def process_and_classify_year(year: int, valid_devices: set, workplace_file_override=None, showed_hours_threshold: int = None):
    """
    Loads daily files, filters by valid devices immediately (saving memory),
    merges with home/work locations, and classifies into Home/Work/Amenity.
    """
    print(f"\n--- Processing Final Classification for {year} ---")
    
    # 1. Load Reference Locations
    home_file = os.path.join(HOME_WORK_LOC_DIR, f'home_location_{year}.csv')
    
    # Workplace logic: Use specific year if available, otherwise fallback/override (per user logic using 2020 for both)
    if workplace_file_override:
        work_file = workplace_file_override
    else:
        work_file = os.path.join(HOME_WORK_LOC_DIR, f'workplace_location_{year}.csv')
        if not os.path.exists(work_file):
             # Fallback to 2020 if 2021 doesn't exist (mimicking user's logic)
             work_file = os.path.join(HOME_WORK_LOC_DIR, 'workplace_location_2020.csv')

    print(f"Loading reference locations:\n Home: {home_file}\n Work: {work_file}")
    
    try:
        home_df = pd.read_csv(home_file)
        work_df = pd.read_csv(work_file)
    except FileNotFoundError as e:
        print(f"Critical Error: Reference file not found. {e}")
        return

    # Add flags
    home_df['home_flag'] = 1
    work_df['workplace_flag'] = 1
    
    # 2. Iteratively Load and Filter Daily Data (Memory Optimized)
    daily_files = glob(os.path.join(OUTPUT_DATA_PATH, 'home_workplace', f'*_{year}_*_home_workplace.csv'))
    # Sort carefully (ensure matching part of filename is used for sort if needed, here just basic sort)
    daily_files.sort()
    
    filtered_chunks = []
    print(f"Loading and filtering {len(daily_files)} daily files...")
    
    for f in tqdm(daily_files):
        try:
            # 只读需要的列
            df = pd.read_csv(f, dtype=DTYPE_DICT, nrows=10000)
            
            # 1. Filter by Hours
            if showed_hours_threshold is not None:
                df = df[df['showed_hours_today'] >= showed_hours_threshold]
            
            # 2. Filter by Valid Devices (Crucial for Memory)
            df = df[df[DEVICE_ID_COL].isin(valid_devices)]
            
            if df.empty: continue
            
            # Drop unnecessary columns to save memory before concat
            cols_to_drop = ['home', 'workplace'] # We will re-merge these from the master list
            df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
            
            filtered_chunks.append(df)
        except Exception as e:
            print(f"Skipping {f}: {e}")
            continue
            
    if not filtered_chunks:
        print(f"No valid data found for {year}.")
        return

    # 3. Concatenate
    full_df = pd.concat(filtered_chunks, ignore_index=True)
    del filtered_chunks
    gc.collect()
    
    # Calculate weighted hours
    full_df['weighted_hours_today'] = (full_df[DURATION_COL] / full_df['showed_hours_today']) * 24
    
    # 4. Merge with Reference Locations
    # Left join to match specific coordinates
    merge_cols = [DEVICE_ID_COL, LAT_COL, LON_COL]
    
    full_df = full_df.merge(home_df[merge_cols + ['home_flag']], on=merge_cols, how='left')
    full_df = full_df.merge(work_df[merge_cols + ['workplace_flag']], on=merge_cols, how='left')
    
    # Fill NAs
    full_df['home_flag'] = full_df['home_flag'].fillna(0)
    full_df['workplace_flag'] = full_df['workplace_flag'].fillna(0)
    
    # 5. Classify
    # Create a 'place' column
    conditions = [
        (full_df['home_flag'] == 1),
        (full_df['workplace_flag'] == 1)
    ]
    choices = ['home', 'workplace']
    full_df['place'] = np.select(conditions, choices, default='amenity')
    
    # 6. Select Columns and Save
    output_cols = [DEVICE_ID_COL, LAT_COL, LON_COL, 'day', 'weighted_hours_today', 'showed_hours_today', 'place']
    final_df = full_df[output_cols]
    
    save_path = os.path.join(HOME_WORK_LOC_DIR, f'total_{year}_selected.csv')
    print(f"Saving final result to {save_path}...")
    final_df.to_csv(save_path, index=False)
    
    # 释放内存
    del full_df, final_df, home_df, work_df
    gc.collect()

def main():
    # 配置
    year_configs = [
        {"year": 2020, "home_only": False, "file_limits": 7},
        {"year": 2021, "home_only": True, "file_limits": 7},
    ]

    # --- Part 1: Process Files (Daily) ---
    all_files_to_process = []
    for config in year_configs:
        year = config["year"]
        file_pattern = INPUT_DATA_PATH_TEMPLATE.format(year=year)
        files_for_year = sorted(glob(file_pattern))[:config.get("file_limits", None)]
        all_files_to_process.extend(files_for_year) # 可在此处切片 [:10] 测试

    run_parallel_processing(all_files_to_process, OUTPUT_DATA_PATH)

    # --- Part 2 & 3: Yearly Aggregation & Day Counts ---
    day_counts_all = {}
    for config in year_configs:
        year = config["year"]
        
        # 聚合
        process_yearly_data_optimized(year, OUTPUT_DATA_PATH, config["home_only"])
        
        # 统计天数
        day_counts = analyze_and_save_day_counts_auto(year, OUTPUT_DATA_PATH)
        if day_counts is not None:
            day_counts_all[year] = day_counts
        
        gc.collect()

    # --- Part 4: Cross-Year Validation & Final Classification (Refactored) ---
    
    if 2020 in day_counts_all and 2021 in day_counts_all:
        # 1. 获取有效设备 ID (Both years >= 4 days)
        valid_devices_set = get_valid_devices_cross_year(day_counts_all[2020], day_counts_all[2021])
        
        # 2. 处理 2020 数据
        # 注意：用户原代码中 2020 和 2021 都使用了 2020 的 workplace 文件。
        # 如果你想让 2021 使用它自己的 workplace 文件（如果存在），可以移除 workplace_file_override 参数
        workplace_file_2020 = os.path.join(HOME_WORK_LOC_DIR, 'workplace_location_2020.csv')
        
        process_and_classify_year(2020, valid_devices_set, workplace_file_override=workplace_file_2020)
        
        # 3. 处理 2021 数据
        process_and_classify_year(2021, valid_devices_set, workplace_file_override=workplace_file_2020)
        
    else:
        print("Skipping Part 4: Day counts data missing for 2020 or 2021.")

    print("\nFull pipeline finished.")


In [ ]:
def process_and_classify_year(year: int, valid_devices: set, workplace_file_override=None, showed_hours_threshold: int = None):
    """
    Loads daily files, filters by valid devices immediately (saving memory),
    merges with home/work locations, and classifies into Home/Work/Amenity.
    """
    print(f"\n--- Processing Final Classification for {year} ---")
    
    # 1. Load Reference Locations
    home_file = os.path.join(HOME_WORK_LOC_DIR, f'home_location_{year}.csv')
    
    # Workplace logic: Use specific year if available, otherwise fallback/override (per user logic using 2020 for both)
    if workplace_file_override:
        work_file = workplace_file_override
    else:
        work_file = os.path.join(HOME_WORK_LOC_DIR, f'workplace_location_{year}.csv')
        if not os.path.exists(work_file):
             # Fallback to 2020 if 2021 doesn't exist (mimicking user's logic)
             work_file = os.path.join(HOME_WORK_LOC_DIR, 'workplace_location_2020.csv')

    print(f"Loading reference locations:\n Home: {home_file}\n Work: {work_file}")
    
    try:
        home_df = pd.read_csv(home_file)
        work_df = pd.read_csv(work_file)
    except FileNotFoundError as e:
        print(f"Critical Error: Reference file not found. {e}")
        return

    # Add flags
    home_df['latitude'] = home_df['latitude'].astype('float32')
    home_df['longitude'] = home_df['longitude'].astype('float32')
    home_df['home_flag'] = 1

    work_df['latitude'] = work_df['latitude'].astype('float32')
    work_df['longitude'] = work_df['longitude'].astype('float32')
    work_df['workplace_flag'] = 1

    # 2. Iteratively Load and Filter Daily Data (Memory Optimized)
    daily_files = glob(os.path.join(OUTPUT_DATA_PATH, 'home_workplace', f'*_{year}_*_home_workplace.csv'))
    # Sort carefully (ensure matching part of filename is used for sort if needed, here just basic sort)
    daily_files.sort()
    
    filtered_chunks = []
    print(f"Loading and filtering {len(daily_files)} daily files...")
    
    for f in tqdm(daily_files):
        try:
            # 只读需要的列
            df = pd.read_csv(f, dtype=DTYPE_DICT, nrows=10000)
            
            # 1. Filter by Hours
            if showed_hours_threshold is not None:
                df = df[df['showed_hours_today'] >= showed_hours_threshold]
            
            # 2. Filter by Valid Devices (Crucial for Memory)
            df = df[df[DEVICE_ID_COL].isin(valid_devices)]
            
            if df.empty: continue
            
            # Drop unnecessary columns to save memory before concat
            cols_to_drop = ['home', 'workplace'] # We will re-merge these from the master list
            df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
            
            filtered_chunks.append(df)
        except Exception as e:
            print(f"Skipping {f}: {e}")
            continue
            
    if not filtered_chunks:
        print(f"No valid data found for {year}.")
        return

    # 3. Concatenate
    full_df = pd.concat(filtered_chunks, ignore_index=True)
    del filtered_chunks
    gc.collect()
    
    # Calculate weighted hours
    full_df['weighted_hours_today'] = (full_df[DURATION_COL] / full_df['showed_hours_today']) * 24
    
    # 4. Merge with Reference Locations
    # Left join to match specific coordinates
    merge_cols = [DEVICE_ID_COL, LAT_COL, LON_COL]
    
    full_df = full_df.merge(home_df[merge_cols + ['home_flag']], on=merge_cols, how='left')
    full_df = full_df.merge(work_df[merge_cols + ['workplace_flag']], on=merge_cols, how='left')
    full_df_probe = full_df.copy()
    # Fill NAs
    full_df['home_flag'] = full_df['home_flag'].fillna(0)
    full_df['workplace_flag'] = full_df['workplace_flag'].fillna(0)
    
    # 5. Classify
    # Create a 'place' column
    conditions = [
        (full_df['home_flag'] == 1),
        (full_df['workplace_flag'] == 1)
    ]
    choices = ['home', 'workplace']
    full_df['place'] = np.select(conditions, choices, default='amenity')
    
    # 6. Select Columns and Save
    output_cols = [DEVICE_ID_COL, LAT_COL, LON_COL, 'day', 'weighted_hours_today', 'showed_hours_today', 'place']
    final_df = full_df[output_cols]
    
    save_path = os.path.join(HOME_WORK_LOC_DIR, f'total_{year}_selected.csv')
    print(f"Saving final result to {save_path}...")
    final_df.to_csv(save_path, index=False)
    
    # 释放内存
    gc.collect()
    return home_df, work_df, full_df_probe


In [ ]:
valid_devices_df = pd.read_csv(os.path.join(HOME_WORK_LOC_DIR, "weekday_weekend_count_2020_2021.csv"))
valid_devices_aid = valid_devices_df[valid_devices_df['device_aid_keep'] == 1]['device_aid'].unique().tolist()

# 2. 处理 2020 数据
# 注意：用户原代码中 2020 和 2021 都使用了 2020 的 workplace 文件。
# 如果你想让 2021 使用它自己的 workplace 文件（如果存在），可以移除 workplace_file_override 参数
workplace_file_2020 = os.path.join(HOME_WORK_LOC_DIR, 'workplace_location_2020.csv')

home_df_2020, work_df_2020, full_df_probe_2020 = process_and_classify_year(2020, valid_devices_aid, workplace_file_override=workplace_file_2020)

# 3. 处理 2021 数据
home_df_2021, work_df_2021, full_df_probe_2021 = process_and_classify_year(2021, valid_devices_aid, workplace_file_override=workplace_file_2020)

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.csv as csv

# 解决 ArrowInvalid: offset overflow 问题
# 直接使用 pyarrow 读取 CSV，并将 string 类型转换为 large_string (支持 >2GB 数据)
table = csv.read_csv("/nas/houce/UK_mobility/data_20251226_new/home_workplace_location/total_2020_selected_for_2021.csv", ncols=1000000)

new_schema = []
for field in table.schema:
    if pa.types.is_string(field.type):
        new_field = field.with_type(pa.large_string())
    else:
        new_field = field
    new_schema.append(new_field)

table = table.cast(pa.schema(new_schema))

#转换回 pandas，使用 arrow 后端
aaa = table.to_pandas(types_mapper=pd.ArrowDtype)
# aaa_select = aaa.iloc[:1000000]

In [ ]:
idx = aaa[aaa['place'] == 'amenity'].groupby(['device_aid', 'day'])['weighted_hours_today'].idxmax()
idx = idx.dropna().astype(int)


In [ ]:
aaa.loc[idx]

In [ ]:
aaa.groupby(['device_aid', 'day'])['weighted_hours_today'].max().reset_index()

In [ ]:
aaa_show_gt_11 = aaa[aaa['showed_hours_today'] > 11]
aaa_show_gt_11['frequency'] = 1

In [ ]:
aaa_home = aaa_show_gt_11[aaa_show_gt_11['place'] == 'home']
aaa_workplace = aaa_show_gt_11[aaa_show_gt_11['place'] == 'workplace']
aaa_amenity = aaa_show_gt_11[aaa_show_gt_11['place'] == 'amenity']

In [ ]:
aaa_amenity

In [ ]:
import pyreadr
poi_data = pyreadr.read_r("/nas/houce/UK_mobility/data_20251226_new/home_workplace_location/England_poi_2020_2021_03.RData")



In [ ]:
import pyreadr
poi_data = pyreadr.read_r("/nas/houce/UK_mobility/data_20251226_new/home_workplace_location/England_poi_2020_2021_2022_03.RData")



In [ ]:
poi_data["England_poi_2022_03"]

In [ ]:
England_poi_2020_03 = poi_data['England_poi_2020_03']
England_poi_2020_03 = England_poi_2020_03.drop(index=574218)

In [ ]:
aaa_amenity['location'] = aaa_amenity['latitude'].astype(str) + "_" + aaa_amenity['longitude'].astype(str)

In [ ]:
aaa_amenity_exists = aaa_amenity[aaa_amenity['location'].isin(England_poi_2020_03['location'].unique())]

In [ ]:
aaa_amenity_exists = aaa_amenity_exists.merge(England_poi_2020_03, on='location', how='left')
aaa_amenity_exists = aaa_amenity_exists[aaa_amenity_exists['poi_total'] > 0]

In [ ]:
aaa_amenity_exists['t1_weighted_hours_today'] = aaa_amenity_exists['weighted_hours_today'] * aaa_amenity_exists['t1_eating_drinking'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t3_weighted_hours_today'] = aaa_amenity_exists['weighted_hours_today'] * aaa_amenity_exists['t3_attractions_entertainment'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t5_weighted_hours_today'] = aaa_amenity_exists['weighted_hours_today'] * aaa_amenity_exists['t5_health'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t6_weighted_hours_today'] = aaa_amenity_exists['weighted_hours_today'] * aaa_amenity_exists['t6_public_infrastructure'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t8_weighted_hours_today'] = aaa_amenity_exists['weighted_hours_today'] * aaa_amenity_exists['t8_reatail'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t9_weighted_hours_today'] = aaa_amenity_exists['weighted_hours_today'] * aaa_amenity_exists['t9_transport'] / aaa_amenity_exists['poi_total']

aaa_amenity_exists['t1_frequency'] = aaa_amenity_exists['frequency'] * aaa_amenity_exists['t1_eating_drinking'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t3_frequency'] = aaa_amenity_exists['frequency'] * aaa_amenity_exists['t3_attractions_entertainment'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t5_frequency'] = aaa_amenity_exists['frequency'] * aaa_amenity_exists['t5_health'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t6_frequency'] = aaa_amenity_exists['frequency'] * aaa_amenity_exists['t6_public_infrastructure'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t8_frequency'] = aaa_amenity_exists['frequency'] * aaa_amenity_exists['t8_reatail'] / aaa_amenity_exists['poi_total']
aaa_amenity_exists['t9_frequency'] = aaa_amenity_exists['frequency'] * aaa_amenity_exists['t9_transport'] / aaa_amenity_exists['poi_total']


In [ ]:
amenity_results = aaa_amenity_exists.groupby(['device_aid','day']).agg({
    't1_weighted_hours_today': 'sum',
    't3_weighted_hours_today': 'sum',
    't5_weighted_hours_today': 'sum',
    't6_weighted_hours_today': 'sum',
    't8_weighted_hours_today': 'sum',
    't9_weighted_hours_today': 'sum',
    't1_frequency': 'sum',
    't3_frequency': 'sum',
    't5_frequency': 'sum',
    't6_frequency': 'sum',
    't8_frequency': 'sum',
    't9_frequency': 'sum'
}).reset_index()

In [ ]:
amenity_results

In [ ]:
aaa_home

In [ ]:
import polars as pl
import pandas as pd
import pyreadr
from pathlib import Path
import pyarrow as pa
import pyarrow.csv as csv
import argparse
import sys
import os

def resolve_files(args):
    """根据输入的 year 和 base_path 确定具体文件路径"""
    year = args['year']
    base_path = args['base_path']
    
    # 匹配流动性文件逻辑
    if year in ["2021", "2022"]:
        mobility_file = base_path / f"total_{year}_selected.csv"
    elif year == "2020_for_2021":
        mobility_file = base_path / "total_2020_selected_for_2021.csv"
    elif year == "2020_for_2022":
        mobility_file = base_path / "total_2020_selected_for_2022.csv"
    else:
        # 虽然 argparse 有 choices，但这里加一层保护
        raise ValueError(f"Unsupported YEAR value: {year}")
    # 确定输出文件
    if args['output'] is not None:
        output_file = args['output']
    else:
        os.makedirs(base_path / "amenity_weighted_hours_and_frequency", exist_ok=True)
        output_home = base_path / "amenity_weighted_hours_and_frequency" / f"home_{year}.csv"
        output_workplace = base_path / "amenity_weighted_hours_and_frequency" / f"workplace_{year}.csv"

    return mobility_file, output_home, output_workplace

args = {
    'year': None,
    "base_path": Path("/nas/houce/UK_mobility/data_20251226_new/home_workplace_location"),
    "output": None
}

for year in ["2021", "2022"]:
# for year in ["2021", "2022", "2020_for_2021", "2020_for_2022"]:
# for year in ["2022"]:
    args["year"] = year
    mobility_path, output_home, output_workplace = resolve_files(args)
    print(f"Processing YEAR={year}")
    print(f"Mobility file: {mobility_path}")
    print(f"Output home file: {output_home}")
    print(f"Output workplace file: {output_workplace}")

    pl.read_csv(mobility_path, infer_schema_length=10000).write_parquet("temp_data.parquet")

    df_pl = pl.read_parquet("temp_data.parquet")
    df_pl_home = df_pl.filter(pl.col('place') == 'home')
    # df_pl = df_pl.filter(pl.col('place') == 'workplace')

    # 转换回 pandas 处理后续复杂逻辑
    df_home = df_pl_home.to_pandas()
    df_home.to_csv(output_home, index=False)

    df_pl_workplace = df_pl.filter(pl.col('place') == 'workplace')
    df_workplace = df_pl_workplace.to_pandas()
    df_workplace.to_csv(output_workplace, index=False)

In [ ]:
df